# LAB 4: Forecasting & Predictive Analytics cho dữ liệu IoT

Bài này dùng UCI Appliances Energy Prediction để dự báo `Appliances` trong bước thời gian kế tiếp. Điểm cần nhớ: Lab 4 không hỏi điểm hiện tại có bất thường không; Lab 4 hỏi giá trị tương lai có thể là bao nhiêu và quyết định vận hành nên thận trọng thế nào.

## 1. Load dữ liệu
Chạy `src/download_data.py` trước nếu muốn tải UCI official dataset. Nếu không có Internet, notebook vẫn chạy bằng sample fallback.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'src'))
from utils import load_dataset, make_supervised_frame, clean_supervised_frame, FEATURE_COLUMNS, time_split, regression_metrics, risk_from_prediction, recommendation_from_risk
import pandas as pd
import numpy as np

df = load_dataset()
df.head()

In [ ]:
df[['date','Appliances','lights','T1','RH_1','T_out']].describe()

## 2. Tạo supervised forecasting table
Ta tạo target tương lai bằng `target_future = Appliances.shift(-1)`. Với UCI, một bước là 10 phút.

In [ ]:
supervised = make_supervised_frame(df, horizon_steps=1, include_target=True)
supervised = clean_supervised_frame(supervised, FEATURE_COLUMNS, require_target=True)
supervised[['date','Appliances','appliances_lag_1','appliances_rolling_mean_6','target_future']].head()

## 3. Chia train/test theo thời gian
Không dùng random split cho chuỗi thời gian vì sẽ làm thông tin tương lai rò rỉ ngược về quá khứ.

In [ ]:
train_df, test_df = time_split(supervised, train_ratio=0.75)
train_df['date'].min(), train_df['date'].max(), test_df['date'].min(), test_df['date'].max()

## 4. Baseline: Last Value và Moving Average
Baseline là hàng rào chống ảo tưởng. Model ML chỉ có ý nghĩa khi được so với quy tắc đơn giản.

In [ ]:
y_test = test_df['target_future']
last_value_pred = test_df['Appliances']
ma6_pred = test_df['appliances_rolling_mean_6']
print('Last value:', regression_metrics(y_test, last_value_pred))
print('Moving average 6:', regression_metrics(y_test, ma6_pred))

## 5. Train Linear Regression và Random Forest
Linear Regression dễ giải thích; Random Forest mạnh hơn nhưng cần tránh coi nó là hộp thần kỳ.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df['target_future']
X_test = test_df[FEATURE_COLUMNS]

lin = Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())])
lin.fit(X_train, y_train)
lin_pred = lin.predict(X_test)
print('Linear Regression:', regression_metrics(y_test, lin_pred))

rf = RandomForestRegressor(n_estimators=120, max_depth=12, min_samples_leaf=3, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print('Random Forest:', regression_metrics(y_test, rf_pred))

## 6. Model nâng cao: GradientBoostingRegressor
Đây là bản demo nâng cao nhẹ theo hướng boosting. Nó là cầu nối để sinh viên hiểu XGBoost/LightGBM nhưng vẫn chạy bằng scikit-learn, không cần cài thêm thư viện nặng.

In [ ]:
hgb = GradientBoostingRegressor(n_estimators=160, learning_rate=0.05, max_depth=3, min_samples_leaf=3, random_state=42)
hgb.fit(X_train, y_train)
hgb_pred = hgb.predict(X_test)
print('Gradient Boosting:', regression_metrics(y_test, hgb_pred))

## 7. Từ forecast sang risk_level và recommendation
Forecast chưa phải quyết định điều khiển. Dự báo cần đi qua risk rule và safety rule.

In [ ]:
thresholds = {'warning': float(np.quantile(y_train, 0.70)), 'high': float(np.quantile(y_train, 0.90)), 'critical': float(np.quantile(y_train, 0.97))}
example_pred = float(hgb_pred[-1])
risk = risk_from_prediction(example_pred, thresholds)
print(example_pred, risk, recommendation_from_risk(risk), thresholds)

## 8. Chạy script tổng hợp
Sau khi hiểu notebook, sinh viên chạy:

```bash
python src/train_forecast.py
python src/plot_results.py
python src/test_api_local.py
```